# Experiment: data collection (run in the SMAC environment, Python 3.10)

This notebook runs a grid of SMAC battles and saves raw per-unit logs. The analysis and SINDy fitting happen in `analysis/fit_models.ipynb`, which runs in the Python 3.12 environment (the SMAC environment can't load a working SciPy on this macOS).

**Experimental design**

| Factor | Levels | Why |
|---|---|---|
| Map (force ratio) | 3m, 8m, 5m_vs_6m, 8m_vs_9m, 10m_vs_11m | Varies team size and the force ratio N_enemy / N_ally, so both sides take heavy losses in some battles. This breaks the collinearity we saw on 8m alone. |
| Focus-fire probability `p_focus` | 0.0, 0.5, 1.0 | Each time an agent shoots, it targets the lowest-health enemy with probability `p_focus`, otherwise a random enemy in range. This turns doctrine into a continuous parameter. |
| Episodes per configuration | `N_EPISODES` | Repeated battles average out the randomness of individual shots. |

Each configuration is saved to its own file. If the run is interrupted, re-running the notebook skips the configurations that are already done.

In [7]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from smac.env import StarCraft2Env

MAPS        = ["3m", "8m", "5m_vs_6m", "8m_vs_9m", "10m_vs_11m"]
FOCUS_PROBS = [0.0, 0.5, 1.0]
N_EPISODES  = 200          # per configuration; start with ~10 to check timing
STEP_MUL    = 2           # game frames per step (default 8); lower = finer time resolution
BASE_SEED   = 0

OUT = Path("data/experiment")
OUT.mkdir(parents=True, exist_ok=True)
N_NO_ATTACK = 6           # actions 0-5 are no-op, stop, N, S, E, W; 6+i attacks enemy i

## Scripted policy

Each agent follows the same simple doctrine:

1. **Dead** → no-op (the only legal action).
2. **An enemy is in range** → attack. With probability `p_focus`, pick the lowest-health enemy in range (focus fire); otherwise pick a random enemy in range.
3. **No enemy in range** → move one step towards the centroid of the living enemies.

All randomness goes through a seeded `rng`, so a configuration can be regenerated.

In [8]:
def enemy_hp(env, action):
    e = env.enemies[action - N_NO_ATTACK]
    return e.health + e.shield


def move_towards_enemies(env, a, avail):
    unit = env.agents[a]
    alive = [e for e in env.enemies.values() if e.health > 0]
    if not alive:
        return int(avail[0])
    cx = np.mean([e.pos.x for e in alive])
    cy = np.mean([e.pos.y for e in alive])
    dx, dy = cx - unit.pos.x, cy - unit.pos.y
    move = (4 if dx > 0 else 5) if abs(dx) > abs(dy) else (2 if dy > 0 else 3)   # E/W or N/S
    return move if move in avail else int(avail[0])


def policy(env, a, p_focus, rng):
    avail = np.nonzero(env.get_avail_agent_actions(a))[0]
    if env.agents[a].health == 0:
        return 0
    attacks = avail[avail >= N_NO_ATTACK]
    if len(attacks):
        if rng.random() < p_focus:
            return int(min(attacks, key=lambda act: enemy_hp(env, act)))
        return int(rng.choice(attacks))
    return move_towards_enemies(env, a, avail)

## Logging

One row per unit per step (the state *before* the step's actions), plus one final row per unit after the battle ends. Aggregates such as team health are computed in the analysis notebook, so nothing needs re-simulating if we later want a new feature.

In [9]:
def log_state(rows, ep, t, env, actions=None):
    for team, units in (("ally", env.agents), ("enemy", env.enemies)):
        for uid, u in units.items():
            a = actions[uid] if (team == "ally" and actions is not None) else None
            rows.append(dict(
                episode_id=ep, t=t, team=team, unit_id=uid,
                health=u.health, shield=u.shield, alive=u.health > 0,
                x=u.pos.x, y=u.pos.y,
                action=a, target_id=(a - N_NO_ATTACK if a is not None and a >= N_NO_ATTACK else None),
            ))


def simulate_battle(env, ep, rng, p_focus):
    rows = []
    env.reset()
    done, t, info = False, 0, {}
    while not done:
        actions = [policy(env, i, p_focus, rng) for i in range(env.n_agents)]
        log_state(rows, ep, t, env, actions)
        _, done, info = env.step(actions)
        t += 1
    log_state(rows, ep, t, env)
    allies_left  = sum(u.health > 0 for u in env.agents.values())
    enemies_left = sum(u.health > 0 for u in env.enemies.values())
    meta = dict(episode_id=ep, battle_won=info.get("battle_won", False),
                timeout=(allies_left > 0 and enemies_left > 0), length=t,
                allies_left=allies_left, enemies_left=enemies_left)
    return rows, meta

## Run the grid

One StarCraft II instance is launched per configuration and reused for all its episodes. Each configuration gets its own game seed and NumPy seed, so the whole dataset can be regenerated.

Run a small `N_EPISODES` first and look at the timing printout before starting the full run.

In [ ]:
def run_config(map_name, p_focus, seed):
    env = StarCraft2Env(map_name=map_name, step_mul=STEP_MUL, seed=seed, window_size_x=160, window_size_y=120)
    env.episode_limit = int(env.episode_limit * 8 / STEP_MUL)
    print(f"{map_name}: episode limit = {env.episode_limit} steps")
    rng = np.random.default_rng(seed)
    all_rows, all_meta = [], []
    try:
        for ep in range(N_EPISODES):
            rows, meta = simulate_battle(env, ep, rng, p_focus)
            all_rows += rows
            all_meta.append(meta)
        n_agents, n_enemies = env.n_agents, env.n_enemies
    finally:
        env.close()

    units = pd.DataFrame(all_rows)
    episodes = pd.DataFrame(all_meta)
    for df in (units, episodes):
        df["map"], df["p_focus"] = map_name, p_focus
    episodes["n_agents"], episodes["n_enemies"] = n_agents, n_enemies
    episodes["seed"], episodes["step_mul"] = seed, STEP_MUL
    return units, episodes


for i, map_name in enumerate(MAPS):
    for j, p in enumerate(FOCUS_PROBS):
        tag = f"{map_name}_p{p:.2f}"
        f_units, f_eps = OUT / f"units_{tag}.csv.gz", OUT / f"episodes_{tag}.csv.gz"
        if f_units.exists() and f_eps.exists():
            print(f"skip {tag} (already done)")
            continue
        t0 = time.time()
        units, episodes = run_config(map_name, p, seed=BASE_SEED + 100 * i + j)
        units.to_csv(f_units, index=False)
        episodes.to_csv(f_eps, index=False)
        print(f"{tag}: {len(episodes)} episodes, win rate {episodes.battle_won.mean():.2f}, "
              f"{time.time() - t0:.0f} s")

3m: episode limit = 240 steps
3m_p0.00: 200 episodes, win rate 0.23, 65 s
3m: episode limit = 240 steps
3m_p0.50: 200 episodes, win rate 0.53, 73 s
3m: episode limit = 240 steps
3m_p1.00: 200 episodes, win rate 0.68, 64 s
skip 8m_p0.00 (already done)
skip 8m_p0.50 (already done)
skip 8m_p1.00 (already done)
5m_vs_6m: episode limit = 280 steps
5m_vs_6m_p0.00: 200 episodes, win rate 0.00, 64 s
5m_vs_6m: episode limit = 280 steps


## Quick check

Win rate, timeouts and battle length for every configuration. If a map shows many timeouts, battles are ending in a chase rather than a fight, which is worth knowing before fitting.

In [ ]:
eps = pd.concat([pd.read_csv(f) for f in sorted(OUT.glob("episodes_*.csv.gz"))], ignore_index=True)
eps.groupby(["map", "p_focus"]).agg(
    episodes=("battle_won", "size"),
    win_rate=("battle_won", "mean"),
    timeouts=("timeout", "mean"),
    mean_length=("length", "mean"),
).round(2)

episodes  win_rate  timeouts  mean_length
map        p_focus                                           
10m_vs_11m 0.0           200      0.00       0.0        83.30
           0.5           200      0.05       0.0       100.28
           1.0           200      0.13       0.0       117.56
3m         0.0           200      0.00       0.0        60.00
           0.5           200      0.00       0.0        60.00
           1.0           200      0.00       0.0        60.00
5m_vs_6m   0.0           200      0.00       0.0        67.58
           0.5           200      0.00       0.0        69.95
           1.0           200      0.00       0.0        70.00
8m         0.0           200      0.38       0.0        99.28
           0.5           200      0.80       0.0       101.64
           1.0           200      0.98       0.0        94.18
8m_vs_9m   0.0           200      0.00       0.0        76.58
           0.5           200      0.03       0.0        91.96
           1.0           200      0.08       0.0       107.01

In [ ]:
eps.groupby("map")[["timeout", "length"]].agg(["mean", "max"])

timeout             length     
              mean    max        mean  max
map                                       
10m_vs_11m     0.0  False  100.380000  150
3m             0.0  False   60.000000   60
5m_vs_6m       0.0  False   69.178333   70
8m             0.0  False   98.368333  120
8m_vs_9m       0.0  False   91.850000  120